# Demo: the linear-programming nonclassicality solver

This notebook demonstrates the primary solver in `src.solvers.lp_solver`.
It is presentation-only: all logic is imported from `src`.

## Problem

A photon-number distribution $\mathbf P = (P_0, P_1, \dots)$ is **classical**
iff it is a mixture of Poissonians. We test this through the witness

$$ f(\boldsymbol\lambda) = \max\!\Big(0, \max_{\mu\ge 0}
\boldsymbol\lambda\cdot\mathbf p(\mu)\Big) - \boldsymbol\lambda\cdot\mathbf P,
\qquad p_k(\mu) = e^{-\mu}\mu^k/k!. $$

Minimising $f$ over $\boldsymbol\lambda$ is a semi-infinite linear program;
a negative optimum (reported as `num > 0`) certifies nonclassicality.

In [ ]:
# Make the project importable when running from the notebooks/ directory.
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from src.config import SOLVER
from src.solvers.lp_solver import solve_semi_infinite_lp
from src.solvers.minimize_solver import minimize_witness
from src.analytical.jaynes_cummings import Pjaycum
from src.utils.numerics import inverse_factorials

## A single probability vector

We evaluate the witness for one distribution with both solvers.

In [ ]:
indx = np.array([0, 1, 2, 3])
probs = np.array([0.6, 0.1, 0.2, 0.05])
fact = inverse_factorials(int(np.max(indx)))
bounds = [SOLVER.lambda_bound] * len(indx)

lp = solve_semi_infinite_lp(probs, indx, fact, bounds)
print('LP num        =', round(lp['num'], 8), '-> nonclassical:', lp['num'] > 0)
print('LP lambda     =', np.round(lp['lam'], 4))

## Jaynes-Cummings photon statistics

The Jaynes-Cummings model gives analytical $P_n(\nu, \tau)$ for a thermal
field (mean photon number $\nu$) coupled to a two-level atom. We plot the
first few $P_n$ versus the scaled interaction time $\tau$.

In [ ]:
tau = np.linspace(0.01, 10.0, 400)
nu = 1.0
pe = 1.0 / 3.0

plt.figure(figsize=(7, 4))
for n in range(4):
    plt.plot(tau, Pjaycum(n=n, nu=nu, tau=tau, pe=pe), label=f'$P_{n}$')
plt.xlabel(r'$\tau$'); plt.ylabel(r'$P_n$')
plt.title(f'Jaynes-Cummings statistics ($\\nu={nu}$, $p_e={pe:.2f}$)')
plt.legend(); plt.tight_layout(); plt.show()